## Quantum State Tomography with JuMP

Quantum state tomography estimates a density matrix from measurement data. This is an optimization problem with physics constraints:

- the state should explain the observed measurement probabilities;
- the trace should be one, because total probability is normalized;
- the density matrix should be positive semidefinite, because all measurement probabilities must be nonnegative.

We use a two-level quantum system so that the density matrix is a real Hermitian `2 × 2` matrix in the core notebook:

$$
\rho = \begin{bmatrix}
\rho_{11} & \rho_{12} \\
\rho_{12} & \rho_{22}
\end{bmatrix}.
$$

The Born rule gives the probability $p_i$ of measuring $E_i$,

$$
p_i = \mathrm{tr}(E_i\rho).
$$

This version intentionally leaves selected implementations as `TODO` exercises. Replace each `error("TODO: ...")` line with working Julia/JuMP code.


**Attribution:** Licensed under <a href="http://creativecommons.org/licenses/by-nc-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">CC BY-NC-SA 4.0<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/nc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1"></a></p>

```text
Andy Goldschmidt 
Andy.Goldschmidt@jhuapl.edu
Johns Hopkins Applied Physics Laboratory
```

## Setup

Run these cells from the `lecture-1-optimization` environment:

```julia
julia --project=.
```


In [ ]:
using GLMakie
using Ipopt
using JuMP
using LinearAlgebra
using DataFrames
using Random

## Problem data and the Born rule

We start with four rank-one projective measurements: the computational basis states $|0\rangle$, $|1\rangle$ and the $X$-basis states $|+\rangle$, $|-\rangle$.

The first core exercise is to compute the model-predicted probabilities using the Born rule. The helper `predicted_probability(E, ρ)` should work both when `ρ` is an ordinary numeric matrix and when `ρ` is a matrix of JuMP decision variables.


In [ ]:
states = Dict(
    "|0⟩" => [1.0, 0.0],
    "|1⟩" => [0.0, 1.0],
    "|+⟩" => normalize([1.0, 1.0]),
    "|−⟩" => normalize([1.0, -1.0]),
)

# Separating the lookup table from the desired measurements
measurement_names = ["|0⟩", "|1⟩", "|+⟩", "|−⟩"]

projector(ψ) = ψ * ψ'

# Example: |0⟩⟨0|
projector(states["|0⟩"])

In [ ]:
# Collect the projectors
measurements = [projector(states[name]) for name in measurement_names]

# A random state, idk
ketθ = normalize([-1.0, 0.55])
ρ_true = projector(ketθ)

In [ ]:
#  states = Dict(
#     "|0⟩" => [1.0, 0.0],
#     "|1⟩" => [0.0, 1.0],
#     "|+⟩" => normalize([1.0, 1.0]),
#     "|−⟩" => normalize([1.0, -1.0]),
# )
function predicted_probability(E, ρ)
    # Born-rule prediction for measuring the projector E on state ρ.
    n = size(ρ, 1)
    return sum(E[i, j] * ρ[j, i] for i in 1:n, j in 1:n)
end

predicted_probability(projector(states["|0⟩"]), projector(normalize([1.0, -1.0]))) 

In [ ]:
probabilities_true = [predicted_probability(E, ρ_true) for E in measurements]

In [92]:
# This fixed perturbation mimics finite-shot measurement noise.
measurement_noise = [0.06, -0.03, 0.08, -0.02]
probabilities_observed = probabilities_true .+ measurement_noise
println(typeof(probabilities_observed))
collect(zip(measurement_names, probabilities_true, probabilities_observed))

# Contrived example for "|0⟩" and "|+⟩":
# probabilities_observed = [0.95, 0.95]


Vector{Float64}


4-element Vector{Tuple{String, Float64, Float64}}:
 ("|0⟩", 0.7677543186180422, 0.8277543186180423)
 ("|1⟩", 0.23224568138195778, 0.20224568138195778)
 ("|+⟩", 0.07773512476007674, 0.15773512476007673)
 ("|−⟩", 0.9222648752399231, 0.902264875239923)

## Core helper functions

The three optimization models below share the same building blocks

Implementing the helpers

`real_density_variables`
`residual_vector`
`add_least_squares_objective!`


 hopefully makes the optimization structure easier to see.

In [ ]:
function real_density_variables(model)
    @variable(model, ρ11)
    @variable(model, ρ12)
    @variable(model, ρ22)

    return [
        ρ11  ρ12
        ρ12  ρ22
    ]
end

function residual_vector(probabilities, measurements, ρ)
    residuals = probabilities .- predicted_probability.(measurements, Ref(ρ))

    return residuals
end

function add_least_squares_objective!(model, residuals)

    @objective(model, Min, sum(residuals.^2))

    return model
end

### Version 1: Tomography as unconstrained least squares

First things first, just fit the observations. It might fit the observations well, but it doesn't have to be a valid density matrix.

In [ ]:
function tomography_model_v1(probabilities, measurements)
    model = Model(Ipopt.Optimizer)

    ρ = real_density_variables(model)
    residuals = residual_vector(probabilities, measurements, ρ)
    add_least_squares_objective!(model, residuals)

    set_silent(model)
    optimize!(model)

    return (
        ρ=value.(ρ), 
        objective=objective_value(model), 
        status=termination_status(model)
    )
end


In [ ]:
ρ_v1, J_v1, status_v1 = tomography_model_v1(probabilities_observed, measurements)
ρ_v1


#### Evaluate the result

[The IBM notes](https://quantum.cloud.ibm.com/learning/en/courses/general-formulation-of-quantum-information/density-matrices/density-matrix-basics) taught us about what counts as a valid density matrix.

There were two properties, which we can calculate using `tr` and `eigvals`. How'd we do?




In [ ]:
println(tr(ρ_v1))
println(eigen(ρ_v1).values)

### Version 2: add the trace constraint

A density matrix needs a constraint on its trace!  JuMP and Ipopt solve the resulting constrained optimization problem for us, but we still need to tell JuMP what the constraint is.




In [ ]:
function add_trace_constraint!(model, ρ)
    @constraint(model, tr(ρ)==1)
    return model
end

function tomography_model_v2(probabilities, measurements)
    model = Model(Ipopt.Optimizer)

    ρ = real_density_variables(model)
    residuals = residual_vector(probabilities, measurements, ρ)
    add_least_squares_objective!(model, residuals)
    add_trace_constraint!(model, ρ)

    set_silent(model)
    optimize!(model)

    return (
        ρ=value.(ρ), 
        objective=objective_value(model), 
        status=termination_status(model)
    )
end


In [ ]:
ρ_v2, J_v2, status_v2 = tomography_model_v2(probabilities_observed, measurements)
ρ_v2


#### Evaluate again

In [ ]:
println(tr(ρ_v2))
println(eigen(ρ_v2).values)

### Version 3: add the positive-semidefinite constraint

Let's solve the physically constrained tomography problem, in all its glory.

The density matrix needs positive eigenvalues. This one turns out to be nonlinear!

In [ ]:


function add_real_2x2_psd_constraints!(model, ρ)
    a, b = ρ[1,1], ρ[1,2]
    d = ρ[2,2]
    @constraint(model, a>=0)
    @constraint(model, d >= 0)
    @constraint(model, a*d - b^2 >= 0)

    return model
end

function tomography_model_v3(probabilities, measurements)
    model = Model(Ipopt.Optimizer)

    ρ = real_density_variables(model)
    residuals = residual_vector(probabilities, measurements, ρ)
    add_least_squares_objective!(model, residuals)
    add_trace_constraint!(model, ρ)
    add_real_2x2_psd_constraints!(model, ρ)

    set_silent(model)
    optimize!(model)

    return value.(ρ), objective_value(model), termination_status(model)
end


In [ ]:
ρ_v3, J_v3, status_v3 = tomography_model_v3(probabilities_observed, measurements)
ρ_v3


### Diagnostics: physicality, reconstruction error, and Bloch vectors

A tomography estimate can have a small least-squares objective and still fail to be a physical density matrix. To diagnose this, compare:

- trace: should be close to `1`;
- minimum eigenvalue: should be nonnegative;
- reconstruction error: distance from the known simulated state `ρ_true`;
- Bloch vector: a geometric representation of the qubit state.

For the real matrices in this core notebook, the Bloch vector has no $y$ component:

$$
\rho = \frac{1}{2}
\begin{bmatrix}
1+z & x \\
x & 1-z
\end{bmatrix},
\qquad
x = 2\rho_{12},\quad y=0,\quad z=\rho_{11}-\rho_{22}.
$$

A physical state should lie inside the Bloch sphere. Pure states lie on the surface.


In [ ]:
function is_physical_density_matrix(ρ; atol = 1e-8) #atol = absolute tolerance
    # TODO: Check whether ρ is a valid density matrix.
    # Hint: check tr(ρ) and minimum(eigvals(ρ)) conditions, with tolerance atol.
    return isapprox(tr(ρ),1.0,atol=atol) && minimum(eigvals(ρ)) >= -atol
end

function reconstruction_error(ρ, ρ_reference)
    # TODO: Compare an estimate against the known reference state.
    # Hint: use a matrix norm.
    return (tr((ρ-ρ_reference)*(ρ-ρ_reference)'))^0.5 #2Norm on the spectrum distributing across difference states
end

function bloch_vector(ρ)
    x = 2*real(ρ[1,2])
    z = real(ρ[1,1]) - ρ[2,2]
    y = 0 #real(p[1,1] - ρ[2,2])
    return [x,y,z]

end

reconstruction_records = [
    (name = "true state", ρ = ρ_true, objective = 0.0, status = missing),
    (name = "v1: unconstrained LS", ρ = ρ_v1, objective = J_v1, status = status_v1),
    (name = "v2: trace constrained", ρ = ρ_v2, objective = J_v2, status = status_v2),
    (name = "v3: trace + PSD constrained", ρ = ρ_v3, objective = J_v3, status = status_v3),
]

# TODO: Build a compact comparison table with DataFrames using the diagnostics above.
reconstruction_summary = DataFrame((name =r.name, x=bloch_vector(r.ρ)[1], y=bloch_vector(r.ρ)[2],z=bloch_vector(r.ρ)[3], objective=r.objective, status=r.status) for r in reconstruction_records)


In [ ]:
using QuantumToolbox

function plot_bloch_vectors(reconstruction_records)
    b = Bloch()
    colors = ["black", "red", "blue", "green"]
    b.view = [30,30]
    b.vector_color = colors[1:length(reconstruction_records)]
    b.vector_width = 0.01

    for r in reconstruction_records
        bv = bloch_vector(r.ρ)
        add_vectors!(b, bv)
    end
    fig, _ = render(b)
    return fig
end



plot_bloch_vectors(reconstruction_records)


---

## Extension projects

The core notebook reconstructs a real two-level density matrix from four projective measurements with fixed additive noise. The projects below each change one modeling assumption. They are designed to connect the same optimization framework to experimental design, complex quantum states, statistical noise models, and explicit linear-algebra solutions.


### Project 1: remove measurements and study identifiability

**What you will do.** Repeat the reconstruction using only subsets of the measurements. For example, try the $Z$ basis only, the $X$ basis only, or one measurement from each basis.

**Why it matters.** Tomography is only possible when the measurements contain enough information to determine the unknown state. If measurements are removed, the optimization problem can become underdetermined: many density matrices may explain the same data. The trace and PSD constraints can reduce the set of possible answers, but they cannot create information that was never measured.

**Questions to answer.**

- Which subsets give a unique-looking reconstruction?
- Which subsets allow many states to fit the data equally well?
- When measurements are missing, do the trace and PSD constraints help or hide the lack of information?
- Can two different physical density matrices produce the same observed probabilities for a chosen subset?


In [64]:
measurement_subsets = Dict(
    "Z basis only" => ["|0⟩", "|1⟩"],
    "X basis only" => ["|+⟩", "|−⟩"],
    "one Z and one X" => ["|0⟩", "|+⟩"],
    "missing |−⟩" => ["|0⟩", "|1⟩", "|+⟩"],
)

function make_measurements(names)
    return [projector(states[name]) for name in names]
end

subset_name = "one Z and one X"
measurements_subset = make_measurements(measurement_subsets[subset_name])
probabilities_true_subset = [predicted_probability(E, ρ_true) for E in measurements_subset]
println(typeof(probabilities_true_subset))
# TODO: Decide how to construct observed probabilities for the subset.
# Option 1: use the noiseless probabilities_true_subset.
# Option 2: add a subset of measurement_noise.
# Option 3: simulate finite-shot data after completing Project 3.
#probabilites_observed_subset = make random error from -0.1-0.1 and add it the probabilties true subset for every element in that measurements_subset
probabilities_observed_subset = probabilities_true_subset .+ rand(length(probabilities_true_subset)) .* 0.2 .- 0.1
println(typeof(probabilities_observed_subset))
#error("TODO: Choose observed data for the measurement subset")

# TODO: Run tomography_model_v1, tomography_model_v2, and tomography_model_v3 on the subset.
# TODO: Compare objectives, physicality, and Bloch vectors across measurement subsets.

ρ_1, J_1, status_1 = tomography_model_v1(probabilities_observed_subset, measurements_subset)
ρ_2, J_2, status_2 = tomography_model_v2(probabilities_observed_subset, measurements_subset)
ρ_3, J_3, status_3 = tomography_model_v3(probabilities_observed_subset, measurements_subset)
b_1 = bloch_vector(ρ_1)
b_2 = bloch_vector(ρ_2)
b_3 = bloch_vector(ρ_3)
records = [
    (name = "true state", ρ = ρ_true, objective = 0.0, status = missing, vector = bloch_vector(ρ_true), physicality = is_physical_density_matrix(ρ_true),error = reconstruction_error(ρ_true,ρ_true)),
    (name = "v1: unconstrained LS", ρ = ρ_1, objective = J_1, status = status_1, vector= b_1, physicality = is_physical_density_matrix(ρ_1), error = reconstruction_error(ρ_1, ρ_true)),
    (name = "v2: trace constrained", ρ = ρ_2, objective = J_2, status = status_2, vector= b_2, physicality = is_physical_density_matrix(ρ_2), error = reconstruction_error(ρ_2, ρ_true)),
    (name = "v3: trace + PSD constrained", ρ = ρ_3, objective = J_3, status = status_3, vector= b_3, physicality = is_physical_density_matrix(ρ_3), error = reconstruction_error(ρ_3,ρ_true)),
]

identifiability_results = DataFrame((name =r.name, x=bloch_vector(r.ρ)[1], y=bloch_vector(r.ρ)[2],z=bloch_vector(r.ρ)[3], objective=r.objective, status=r.status, vector=r.vector, physicality = r.physicality, error = r.error) for r in records)






Vector{Float64}
Vector{Float64}


Row,name,x,y,z,objective,status,vector,physicality,error
,String,Float64,Float64,Float64,Float64,Terminat…?,Array…,Bool,Float64
1,true state,-0.84453,0.0,0.535509,0.0,missing,"[-0.84453, 0.0, 0.535509]",true,0.0
2,v1: unconstrained LS,-0.715501,0.0,1.02991,-4.85723e-17,LOCALLY_SOLVED,"[-0.715501, 0.0, 1.02991]",false,0.429279
3,v2: trace constrained,-1.04334,0.0,0.702077,1.38778e-16,LOCALLY_SOLVED,"[-1.04334, 0.0, 0.702077]",false,0.183398
4,v3: trace + PSD constrained,-0.82965,0.0,0.558284,0.0165847,LOCALLY_SOLVED,"[-0.82965, 0.0, 0.558284]",true,0.019237


In [ ]:
plot_bloch_vectors(records)

### Project 2: add Y-basis measurements and complex Hermitian density matrices

**What you will do.** Generalize the notebook from real density matrices to complex Hermitian density matrices by adding the Pauli-$Y$ basis measurements:

$$
|+i\rangle = \frac{1}{\sqrt{2}}\begin{bmatrix}1 \\ i\end{bmatrix},
\qquad
|-i\rangle = \frac{1}{\sqrt{2}}\begin{bmatrix}1 \\ -i\end{bmatrix}.
$$

The density matrix becomes

$$
\rho =
\begin{bmatrix}
\rho_{11} & a + ib \\
a - ib & \rho_{22}
\end{bmatrix}.
$$

**Why it matters.** The real core notebook can only represent states with Bloch vectors in the $x$-$z$ plane. A general qubit also has a $y$ component, which appears as imaginary coherence in the off-diagonal entries. Without $Y$-basis measurements, that imaginary part is not identifiable.

**Questions to answer.**

- What changes in the Born-rule prediction when measurement matrices are complex?
- How does the PSD determinant constraint change?
- Can the original $X/Z$ measurements recover a state with nonzero imaginary coherence?
- What happens to the Bloch vectors after adding the $Y$-basis measurements?


In [111]:
states_complex = Dict(
    "|0⟩" => ComplexF64[1.0, 0.0],
    "|1⟩" => ComplexF64[0.0, 1.0],
    "|+⟩" => normalize(ComplexF64[1.0, 1.0]),
    "|−⟩" => normalize(ComplexF64[1.0, -1.0]),
    "|+i⟩" => normalize(ComplexF64[1.0, 1.0im]),
    "|−i⟩" => normalize(ComplexF64[1.0, -1.0im]),
)

projector_complex(ψ) = ψ * ψ'

measurement_names_complex = ["|0⟩", "|1⟩", "|+⟩", "|−⟩", "|+i⟩", "|−i⟩"]
measurements_complex = [projector_complex(states_complex[name]) for name in measurement_names_complex]

ketφ = normalize(ComplexF64[1.0, 0.4 + 0.7im])
ρ_true_complex = projector_complex(ketφ)
probabilities_true_complex = [real(tr(E * ρ_true_complex)) for E in measurements_complex]

function tomography_model_complex(probabilities, measurements)
    model = Model(Ipopt.Optimizer)

    @variable(model, ρ11)
    @variable(model, ρ22)
    @variable(model, ρ12_re)
    @variable(model, ρ12_im)

    # TODO: Build predicted probabilities for the complex Hermitian matrix.

    
    predicted = [real(predicted_probability(E, [ρ11 ρ12_re+im*ρ12_im; ρ12_re-im*ρ12_im ρ22])) for E in measurements]
    residuals = predicted .- probabilities

    # TODO: Add least-squares or MLE objective.
    #error("TODO: Complete tomography_model_complex")
    @objective(model, Min, sum(residuals.^2))

    set_silent(model)
    optimize!(model)

    ρ_estimate = [
        value(ρ11)               value(ρ12_re) + im * value(ρ12_im)
        value(ρ12_re) - im * value(ρ12_im)   value(ρ22)
    ]
    #println(objective_function(model))

    return ρ_estimate, objective_value(model), termination_status(model)
end

# TODO: Reconstruct ρ_true_complex from probabilities_true_complex or noisy observations.
println(ρ_true_complex)
ρ_complex, J_complex, status_complex = tomography_model_complex(probabilities_true_complex, measurements_complex)

#measurement_bloch_vectors = [Bloch_vector(projector_complex(states_complex[name])) for name in measurement_names_complex]



ComplexF64[0.6060606060606061 + 0.0im 0.24242424242424246 - 0.42424242424242425im; 0.24242424242424246 + 0.42424242424242425im 0.3939393939393939 + 0.0im]


(ComplexF64[0.606060606060606 + 0.0im 0.24242424242424251 - 0.4242424242424242im; 0.24242424242424251 + 0.4242424242424242im 0.39393939393939387 + 0.0im], -5.551115123125783e-17, LOCALLY_SOLVED)

### Project 3: simulate finite-shot count noise and use MLE / cross-entropy

**What you will do.** Replace fixed additive probability noise with simulated measurement counts. Then compare least squares on empirical frequencies with maximum likelihood estimation on the counts.

For a yes/no measurement with $N_i$ shots and $k_i$ observed successes,

$$
k_i \sim \mathrm{Binomial}(N_i, q_i),
\qquad
q_i = \mathrm{tr}(E_i\rho).
$$

The negative log-likelihood, ignoring constants that do not depend on $\rho$, is

$$
-\sum_i \left[k_i\log(q_i) + (N_i-k_i)\log(1-q_i)\right].
$$

This is the binary cross-entropy loss.

**Why it matters.** Least squares treats all probability errors symmetrically and does not know how many shots produced each estimate. MLE uses the count model directly and usually behaves better when probabilities are near 0 or 1, or when different measurements have different shot counts.

**Questions to answer.**

- How do least squares and MLE compare when the number of shots is small?
- How many shots are needed before the two reconstructions look similar?
- What numerical safeguards are needed when $q_i$ is close to `0` or `1`?
- How does the PSD constraint interact with noisy count data?


In [ ]:
function binomial_count(shots, p)
    return count(rand() < p for _ in 1:shots)
end

Random.seed!(1234)
shots = fill(100, length(probabilities_true))
counts = [binomial_count(shots[i], probabilities_true[i]) for i in eachindex(shots)]
frequencies = counts ./ shots
#less shots more erors
collect(zip(measurement_names, counts, shots, frequencies))
#MLE finds thr parameters that make observed data most probable
#cross entropy measurws the difference between true and predicted proaility distributions 
#minimizing quantum crosss-entropy = maximizing quantum likelihood
#

4-element Vector{Tuple{String, Int64, Int64, Float64}}:
 ("|0⟩", 7716, 10000, 0.7716)
 ("|1⟩", 2329, 10000, 0.2329)
 ("|+⟩", 718, 10000, 0.0718)
 ("|−⟩", 9281, 10000, 0.9281)

In [114]:
function tomography_model_mle(counts, shots, measurements)
    model = Model(Ipopt.Optimizer)

    ρ = real_density_variables(model)
    set_start_value(ρ[1,1], 0.6)
    set_start_value(ρ[2,2], 0.4)
    set_start_value(ρ[1,2], 0.05)
    # rho came back exactly equalt to whatever start vlaue youd given it meaning optimize! never actually iterated at all Ipopt rejected the problem outright
    # q between 0 and 1 and logq[i] and log(1-q[i])
    #when they are all the same number q- is 0 then log(q-) = infit 
    #where q = the predicted probability of getting a yes utocme for a given measurment 
    q = [predicted_probability(E, ρ) for E in measurements]

    add_trace_constraint!(model, ρ)
    add_real_2x2_psd_constraints!(model, ρ)

    ϵ = 1e-6

    # TODO: Constrain every predicted probability q[i] to stay in [ϵ, 1 - ϵ].
    # This prevents log(0) in the likelihood.
    #error("TODO: Add probability-domain constraints")

    for i in eachindex(q)
        @constraint(model, q[i] >= ϵ )
        @constraint(model, q[i] <= 1-ϵ )
    end


    # TODO: Add the negative binomial log-likelihood / binary cross-entropy objective:
    #   -sum(counts[i] * log(q[i]) + (shots[i] - counts[i]) * log(1 - q[i]) for i in eachindex(counts))
    #error("TODO: Add MLE / cross-entropy objective")
    @objective(model, Min, -sum(counts[i]*log(q[i]) + (shots[i]-counts[i])*log(1-q[i]) for i in eachindex(counts)))
    #finding the rho that makes your actually obseveredc counts a s probable as possible 
    #println(objective_function(model))
    set_silent(model)
    optimize!(model)

    return value.(ρ), objective_value(model), termination_status(model)
end

# TODO: Compare least squares on frequencies with MLE on counts.
ρ_ls_counts, J_ls_counts, status_ls_counts = tomography_model_v3(frequencies, measurements)

#small j_ls_counts => fitted rho matches the observed dat very closely | large rho there is some noise 



([0.7662466367365585 -0.4232169976127711; -0.4232169976127711 0.23375336326344157], 7.806075219457531e-5, LOCALLY_SOLVED)

In [115]:
ρ_mle, J_mle, status_mle = tomography_model_mle(counts, shots, measurements)

([0.7638354917759221 -0.4247244308987196; -0.4247244308987196 0.23616450822407792], 15972.5807519112, LOCALLY_SOLVED)

In [81]:
ρ_true

2×2 Matrix{Float64}:
  0.767754  -0.422265
 -0.422265   0.232246

In [ ]:
measurment_p3 = Dict(
    "Y basis only"  => ["|+i⟩", "|-i⟩"],
    "one X and one Y" => ["|+⟩", "|+i⟩"],
    "one Z and one Y" => ["|0⟩", "|+i⟩"],
    "all six"  => ["|0⟩", "|1⟩", "|+⟩", "|-⟩", "|+i⟩", "|-i⟩"]
) #x and y mizzing z msiing |-i>

function make_measurments_complex(names)
    return [projectoor_complex(states_complex[name]) for name in names]

subset_name = "Y basis only"
probabilities_true_subset = [real(predicted_probability(E, \ρ_true_complex)) for E in measurements_subset]
Random.seed!(1234)
shots_subset = fill(100, length(probabilities_true_subset[i]))
counts_subset = [binomial_count(shots_subset[i], probabilities_true_subset[i]) for i in eachindex(shots_subset)]
frequencies_subset = counts_subset ./ shots_subset


### Project 4: solve the trace-constrained model by explicit KKT

**What you will do.** Derive and solve the trace-constrained least-squares problem directly as a linear algebra system, then compare the result to `tomography_model_v2`.

For a real symmetric matrix, write the unknowns as

$$
x =
\begin{bmatrix}
\rho_{11} \\
\rho_{12} \\
\rho_{22}
\end{bmatrix}.
$$

Each measurement gives one row of a design matrix $A$:

$$
\mathrm{tr}(E_i\rho)
=
E_{i,11}\rho_{11}
+
(E_{i,12}+E_{i,21})\rho_{12}
+
E_{i,22}\rho_{22}.
$$

The trace constraint (remember, trace is linear, so it has a matrix!) is

$$
c^\top x = 1,
\qquad
c = \begin{bmatrix}1 \\ 0 \\ 1\end{bmatrix}.
$$

The KKT system is

$$
\begin{bmatrix}
2A^\top A & c \\
c^\top & 0
\end{bmatrix}
\begin{bmatrix}
x \\
\lambda
\end{bmatrix}
=
\begin{bmatrix}
2A^\top \hat{p} \\
1
\end{bmatrix}.
$$

**Why it matters.** This project shows what the optimization solver is doing in one special case. It connects JuMP modeling back to linear algebra, normal equations, and Lagrange multipliers.

**Questions to answer.**

- Does the KKT solution match `tomography_model_v2`?
- What happens when measurements are removed and $A$ loses rank?
- Why does this explicit KKT method not directly handle the PSD constraint from Version 3?


In [ ]:
#can we solve v2 without using an optimizer | derive the equations using KKT 
#kkt comines MLE eror and enforicng trace constrints into 1 linear sytstem
function design_matrix_real(measurements)
    # TODO: Build the matrix A whose rows map
    # x = [ρ11, ρ12, ρ22] to predicted probabilities.
    #error("TODO: Implement design_matrix_real")
    A = zeros(length(measurements),3)

    for(i, E) in enumerate(measurements)
        A[i, :] = [
            E[1,1]
            E[1,2] + E[2,1]
            E[2,2]
        ]
    end
    return A
end

function solve_trace_constrained_kkt(probabilities, measurements)
    A = design_matrix_real(measurements)
    c = [1.0, 0.0, 1.0]
    println(length(A))

    ATA = 2*(transpose(A)*A)
    ATp = 2*(transpose(A) *probabilities)
    KKT = [
        ATA c
        c' 0.0
    ]
    rhs = vcat(ATp, 1.0)

    # TODO: Build and solve the KKT system.
    # Hint: use block matrices and the backslash operator.
    solution = KKT \ rhs

    x = solution[1:3]
    λ = solution[4]
    #println(typeof(ATp))
    
    ρ = [
        x[1]  x[2]
        x[2]  x[3]
    ]

    return ρ, λ #tells you how much the optimal value of your objective function would improve if you were to relax that specific constraint by a unti
end


ρ_kkt, λ_kkt = solve_trace_constrained_kkt(probabilities_observed, measurements)
# TODO: Compare ρ_kkt with ρ_v2.
comparison_to_jump = isapprox(ρ_kkt, ρ_v2; atol = 1e-8)
println("KKT matches JuMP: ", comparison_to_jump)
println("KKT Solution:")
println(ρ_kkt)
println("version 2 solution:")
println(ρ_v2)


12
Vector{Float64}
KKT matches JuMP: true
KKT Solution:
[0.8127543186180421 -0.37226487523992324; -0.37226487523992324 0.18724568138195785]
version 2 solution:
[0.8127543186180421 -0.37226487523992324; -0.37226487523992324 0.1872456813819578]


In [ ]:
measurments_small = measurements[1:2]
append!(measurements[1:2], measurements[4:4])
probabilites_small = probabilities_observed[2:4]
println(ρ_true)
ρ_small, λ_small = solve_trace_constrained_kkt(probabilites_small, measurments_small)

#semidefinite desnity matrix inequality is not lnear so u cant make asimple linear KKT modle if u enfore PSD 

[0.7677543186180422 -0.42226487523992323; -0.42226487523992323 0.23224568138195778]
6


LinearAlgebra.SingularException: SingularException(2)